## 第15章　随机变量与分布 —— 用采样代替公式

> 本章目标：理解随机变量的本质——值由概率决定的变量。掌握离散（PMF）和连续（PDF）两种分布类型，亲手模拟正态分布、均匀分布、伯努利分布，并通过极大似然估计（MLE）的直觉建立"从数据反推参数"的思维。最后掌握四种采样策略——Greedy、Temperature、Top-k、Top-p——这是 LLM 文本生成的最后一公里。
> 前置知识：第 1 章（NumPy/matplotlib）、第 3 章（标量与向量）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print('环境就绪 - NumPy + Matplotlib')



### 15.1　离散随机变量 —— 掷骰子与概率质量函数

掷一个公平骰子，结果是几？你无法确定——但你知道每个面出现的概率各是 1/6。骰子的点数就是一个**随机变量**：它的取值由概率决定，而不是由公式确定。

对于可以逐个列出所有可能取值的离散情况，我们用**概率质量函数（PMF）**来描述：P(X=x) 直接给出 X 恰好等于 x 的概率。所有概率之和必须等于 1，每个概率 ≥ 0。

公平骰子掷 600 次，你"应该"看到约 100 次 1、100 次 2……但"大约"到什么程度？我们直接模拟——从小样本（30 次）到大样本（3000 次），亲眼看到频率如何收敛到理论概率 1/6。

📐 **定义　离散随机变量（Discrete Random Variable）**：取值集合 {x₁, x₂, ...} 有限或可数，每个 xᵢ 有概率 pᵢ = P(X=xᵢ)，满足 Σpᵢ = 1 且 pᵢ ≥ 0。

💻 **代码　掷骰子：从小样本到大样本的频率收敛**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, n in enumerate([30, 300, 3000]):
    rolls = np.random.randint(1, 7, size=n)
    # 统计每个面的频率
    freqs = np.array([np.sum(rolls == k) / n for k in range(1, 7)])

    axes[i].bar(range(1, 7), freqs, color='steelblue', edgecolor='white')
    axes[i].axhline(y=1/6, color='red', linestyle='--', linewidth=2, label='理论概率 1/6')
    axes[i].set_title(f'n = {n} 次'); axes[i].set_xlabel('点数')
    axes[i].set_ylabel('频率'); axes[i].set_ylim(0, 0.4)
    axes[i].legend(fontsize=8)

plt.suptitle('掷骰子频率随试验次数收敛到 1/6', fontsize=13)
plt.tight_layout(); plt.show()
print("规律: n 越大，柱高越接近红线——这就是大数定律（第 18 章）")




> **关键洞察**：30 次时柱高参差不齐（偶然性主导），300 次时趋近但仍有波动，3000 次时几乎恰好是 1/6。收敛不是魔法——是大量独立试验的平均效应抹平了随机波动。这个现象在第 18 章会被严格证明为"大数定律"。

🔗 **AI 连接**：分类模型的 softmax 输出就是一个离散概率分布——"猫 0.7, 狗 0.3"恰好是 P(class|image) 的 PMF。第 29 章 Transformer 每步预测下一个 token 时输出一个 50000 维的概率向量——那也是 PMF。

---

### 15.2　连续随机变量 —— 正态分布与概率密度函数

身高 165.3cm、温度 23.7°C——这类可以在区间内取任意值的量是**连续随机变量**。对于连续变量，P(X=精确值)=0（恰好 165.30000... 的概率无限小），所以我们改用**概率密度函数（PDF）**：曲线下某段区间的面积 = X 落在该区间的概率。

正态分布（Normal/Gaussian）是自然界和 AI 中最常见的连续分布。它完全由两个参数决定：**μ（均值/中心）** 和 **σ（标准差/宽度）**。

**68-95-99.7 规则**：约 68% 的数据落在 μ±1σ，约 95% 在 μ±2σ，约 99.7% 在 μ±3σ。偏离 3σ 以上的点出现概率仅约 0.3%——这就是异常检测中"3-sigma 原则"的理论基础。

💻 **代码　正态分布采样 + PDF 叠加 + 68-95-99.7 验证**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf

def normal_pdf(x, mu=0, sigma=1):
    z = (x - mu) / sigma
    return np.exp(-0.5 * z**2) / (sigma * np.sqrt(2 * np.pi))

def normal_cdf(x):
    return 0.5 * (1 + erf(x / np.sqrt(2)))

np.random.seed(42)
mu, sigma = 0.0, 1.0
samples = np.random.randn(10000) * sigma + mu

plt.figure(figsize=(10, 5))
plt.hist(samples, bins=50, density=True, alpha=0.6, color='steelblue',
         edgecolor='white', label='采样直方图 (n=10000)')

x = np.linspace(-4, 4, 200)
plt.plot(x, normal_pdf(x, mu, sigma), 'r-', linewidth=2.5, label=f'N({mu},{sigma}²) PDF')

# 标注 ±kσ 区域
for k, color in zip([1, 2, 3], ['orange', 'green', 'purple']):
    plt.axvspan(mu - k*sigma, mu + k*sigma, alpha=0.08, color=color,
                label=f'±{k}σ ({normal_cdf(k)-normal_cdf(-k):.1%})')

plt.xlabel('x'); plt.ylabel('概率密度'); plt.title('标准正态分布 N(0,1)')
plt.legend(fontsize=8); plt.show()

for k in [1, 2, 3]:
    actual = np.mean(np.abs(samples) < k)
    theory = normal_cdf(k) - normal_cdf(-k)
    print(f"±{k}σ: 采样={actual:.3f}  理论={theory:.3f}  ✓")




> **关键洞察**：正态分布是 AI 训练体系的"默认噪声模型"。权重初始化（第 25 章）用截断正态打破对称性；BatchNorm（第 21 章）假设激活值近似正态来归一化；VAE 的隐变量也假设服从正态分布。

🔗 **AI 连接**：第 18 章中心极限定理揭示为什么正态分布无处不在——大量独立随机变量之和趋向正态，这正是 Mini-Batch SGD 梯度估计的理论依据。

---

### 15.3　均匀分布 —— 最公平的分布

📐 **定义　均匀分布（Uniform Distribution）**：X ~ Uniform(a, b)，PDF 在 [a, b] 内为常数 1/(b−a)，区间外为 0。每个值"等可能"——最公平的连续分布。

在 AI 中，均匀分布最重要的用途是**权重初始化**。训练开始时我们对每个参数一无所知——用均匀分布随机赋值是最自然的"无偏好"起点。PyTorch 的 `nn.Linear` 默认用 `kaiming_uniform_` 初始化权重矩阵。

💻 **代码　Xavier 均匀初始化：权重分布的均值、方差验证**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

fan_in, fan_out = 784, 256
limit = np.sqrt(6 / (fan_in + fan_out))  # Xavier 限界
weights = np.random.uniform(-limit, limit, size=(fan_in, fan_out))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(weights.flatten(), bins=50, density=True, color='steelblue', edgecolor='white')
axes[0].axhline(y=1/(2*limit), color='red', linestyle='--', lw=2,
                label=f'理论 PDF = 1/(2×{limit:.4f})')
axes[0].set_title(f'Xavier Uniform 初始化 ({fan_in}×{fan_out})')
axes[0].legend()

theo_var = (2*limit)**2 / 12  # Uniform[-L,L] 方差 = (2L)²/12
axes[1].bar(['理论方差', '实际方差'], [theo_var, weights.var()],
            color=['lightgray', 'steelblue'], edgecolor='white')
axes[1].set_ylabel('方差'); axes[1].set_title('方差对比')

plt.tight_layout(); plt.show()
print(f"权重均值: {weights.mean():.6f} (理想=0)")
print(f"权重标准差: {weights.std():.6f}")
print(f"范围: [{weights.min():.4f}, {weights.max():.4f}]")




🔗 **AI 连接**：权重初始化（第 25 章）的好坏决定训练能否起步——分布选错，100 层网络的激活值要么爆炸要么消失。

---

### 15.4　伯努利分布 —— 二选一

📐 **定义　伯努利分布（Bernoulli Distribution）**：只有 0 和 1 两种结果，P(X=1)=p, P(X=0)=1−p。是一切"是/否"问题的数学模型。

应用遍布 AI：二分类的输出（P(点击)=0.3）、Dropout（以概率 p 丢弃神经元）、A/B 测试的转化率。n 次独立伯努利之和服从二项分布 Binomial(n, p)。

💻 **代码　伯努利试验 + 二项分布可视化**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
p = 0.3
n_trials = 1000
samples = (np.random.random(n_trials) < p).astype(int)

# 每组 20 次伯努利之和 → 二项分布
n_groups = 50
group_sums = samples[:n_groups*20].reshape(n_groups, 20).sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['失败(0)', '成功(1)'], [np.mean(samples==0), np.mean(samples==1)],
            color=['lightcoral', 'steelblue'], edgecolor='white')
axes[0].set_title(f'Bernoulli(p={p}), n={n_trials}')

axes[1].hist(group_sums, bins=range(0, 22), density=True, alpha=0.7,
             color='steelblue', edgecolor='white')
axes[1].set_xlabel('20次中成功次数'); axes[1].set_title(f'20×Bernoulli ~ Binomial(20,{p})')
plt.tight_layout(); plt.show()
print(f"样本均值={samples.mean():.3f} (理论 p={p})")




---

### 15.5　极大似然估计 —— 让数据"最可能"出现

你掷一枚硬币 10 次，7 次正面。你认为这枚硬币的正面概率 p 是多少？直觉告诉你 p ≈ 0.7——因为 p=0.7 使得"7 正 3 反"这个观察结果**最可能**发生。这就是**极大似然估计（MLE）**的完整直觉。

📐 **定义　极大似然估计（MLE）**：θ̂ = argmax P(data | θ)。选择使观测数据出现概率最大的参数。对正态分布 N(μ, σ²)：μ̂ = Σxᵢ/n（样本均值），σ̂² = Σ(xᵢ−μ̂)²/n（样本方差）。

在深度学习中，**交叉熵损失就是在做 MLE**（第 19 章揭示这个等价性）。每一个 `nn.CrossEntropyLoss()` 底层都在执行极大似然估计。

💻 **代码　MLE 演示：数据越多，μ̂ 越逼近真实 μ**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
true_mu, true_sigma = 5.0, 2.0
n_total = 200
data = np.random.randn(n_total) * true_sigma + true_mu
cum_means = np.cumsum(data) / np.arange(1, n_total+1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 左: μ̂ 收敛曲线
axes[0].axhline(y=true_mu, color='red', ls='--', lw=2, label=f'真实 μ={true_mu}')
axes[0].plot(range(1, n_total+1), cum_means, color='steelblue', lw=1.5)
axes[0].fill_between(range(1, n_total+1),
                     true_mu - true_sigma/np.sqrt(np.arange(1, n_total+1)),
                     true_mu + true_sigma/np.sqrt(np.arange(1, n_total+1)),
                     alpha=0.15, color='steelblue', label='理论 ±σ/√n')
axes[0].set_xlabel('样本量 n'); axes[0].set_ylabel('μ̂')
axes[0].set_title('MLE: μ̂ → true μ  as n → ∞'); axes[0].legend()

# 右: 似然函数随 n 增大变尖锐
for n, color in [(5, 'coral'), (200, 'steelblue')]:
    sub = data[:n]
    mu_grid = np.linspace(true_mu-2, true_mu+2, 200)
    log_lik = -0.5 * np.sum((sub[:,None] - mu_grid[None,:])**2, axis=0) / true_sigma**2
    log_lik = log_lik - log_lik.max()
    axes[1].plot(mu_grid, log_lik, color=color, lw=2, label=f'n={n}, μ̂={sub.mean():.3f}')
    axes[1].axvline(x=sub.mean(), color=color, ls='--', alpha=0.5)
axes[1].axvline(x=true_mu, color='red', ls='-', alpha=0.6, label=f'真实 μ={true_mu}')
axes[1].set_xlabel('μ'); axes[1].set_ylabel('对数似然 (归一化)')
axes[1].set_title('似然函数随样本量增大变尖锐'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"n=5:   μ̂={data[:5].mean():.3f}")
print(f"n=200: μ̂={data.mean():.3f} (真实={true_mu})")




> **关键洞察**：n=5 时 μ̂ 可能是 3.8 或 6.2——不确定；n=200 时 μ̂ 锁定在 4.9~5.1——精确。数据越多，MLE 越可靠。这就是为什么大数据对 AI 如此重要。

🔗 **AI 连接**：第 19 章揭示 MLE ⇔ 最小化交叉熵，第 30 章用这个等价性训练 GPT-2。

---

### 15.6　采样策略 —— LLM 文本生成的"最后一公里"

有了概率分布后怎么"选一个"？这个问题在 LLM 文本生成中至关重要——同样的模型、同样的 prompt，不同采样策略能产生"妙语连珠"和"胡言乱语"两种截然不同的输出。

#### 15.6.1　Greedy（永远选概率最高的）

`argmax(p)`。对于语言模型，每次都选概率最高的下一个 token。**问题**：导致文本循环重复——模型陷入"最安全但最无聊"的循环。GPT-2 做 greedy decoding 经常输出"我不 知道 我不 知道 我不 知道..."。

#### 15.6.2　Temperature（控制"创造力"）

对 logits 除以温度 T 再 softmax。**T→0** 退化为 greedy，**T=1** 标准 softmax，**T→∞** 趋近均匀分布。T<1 更保守（概率集中在高概率 token），T>1 更多样（概率更平滑）。实践中 0.7~0.9 是常见甜点区间。

#### 15.6.3　Top-k（固定候选集大小）

只从概率前 k 高的 token 中采样，其余置零后重归一化。k=50 是常见默认值。**问题**：分布集中时 k=50 太宽松，分布分散时 k=50 又太严格——固定 k 不能自适应。

#### 15.6.4　Top-p / 核采样（动态候选集）

不固定候选数，而是从概率最高的 token 开始累加，直到累积概率 ≥ p（如 p=0.9）。分布集中时候选少，分散时候选多——**p 值自适应决定候选集大小**。Top-p=0.9 是 LLM 工业标准默认值。

💻 **代码　四种采样策略在同一分布上的完整对比**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# 模拟语言模型输出的 logits（10 个 token 的分数）
logits = np.array([3.0, 2.0, 1.5, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01])

def softmax(x, T=1.0):
    x = np.array(x, dtype=np.float64) / T
    return np.exp(x - x.max()) / np.exp(x - x.max()).sum()

orig_probs = softmax(logits, T=1.0)
labels = [f'Token_{chr(65+i)}' for i in range(10)]
colors = plt.cm.viridis(np.linspace(0.15, 0.85, 10))

# 构建五种策略的概率分布
strategies = {}
strategies['Greedy'] = np.eye(10)[np.argmax(orig_probs)]
strategies['Temperature\nT=0.5'] = softmax(logits, T=0.5)
strategies['Temperature\nT=2.0'] = softmax(logits, T=2.0)

topk_probs = orig_probs.copy()
topk_probs[np.argsort(topk_probs)[:-3]] = 0
strategies['Top-k\nk=3'] = topk_probs / topk_probs.sum()

p = 0.9; si = np.argsort(orig_probs)[::-1]
cut = np.searchsorted(np.cumsum(orig_probs[si]), p) + 1
topp = orig_probs.copy(); topp[si[cut:]] = 0
strategies['Top-p\np=0.9'] = topp / topp.sum()

# 可视化
fig, axes = plt.subplots(2, 3, figsize=(15, 8)); axes = axes.flatten()
axes[0].bar(labels, orig_probs, color=colors, edgecolor='white')
axes[0].set_title('原始分布 (T=1.0)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

for ax, (name, probs) in zip(axes[1:], strategies.items()):
    ax.bar(labels, probs, color=colors, edgecolor='white')
    ax.set_title(name, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('同一组 logits 在五种采样策略下的概率分布', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

# 实际采样对比
print("\n各策略采样 1000 次 Top-3 频率:")
for name, probs in strategies.items():
    if 'Greedy' in name:
        freq = strategies['Greedy']
    else:
        samples = np.random.choice(10, size=1000, p=probs)
        freq = np.bincount(samples, minlength=10) / 1000
    top3 = np.argsort(freq)[-3:][::-1]
    s = ', '.join([f'{labels[i]}={freq[i]:.1%}' for i in top3])
    print(f"  {name:18s}: {s}")




> **关键洞察**：没有一种策略最优——Temperature 控制"创造力"，Top-k/Top-p 划定"候选范围"。**组合使用 Temperature + Top-p 是 GPT-4 和 Claude 的默认行为**。第 30.3 节将用 GPT-2 在同一 prompt 上对比这四种策略的实际生成效果——greedy 循环废话、高温度胡言乱语、top-p=0.9 产出行云流水的文本。

🔗 **AI 连接**：第 30 章 GPT-2 文本生成实战中，`model.generate(do_sample=True, temperature=0.7, top_p=0.9)` 三个参数就来自本节。

> 💡 **Agent 视角框**
>
> 在 Agent 系统中，采样策略直接决定行为风格：
>
> - **工具调用（Function Calling）** → 低温度（0.1~0.3），保证动作确定性
> - **创意规划（CoT）** → 中等温度（0.7~0.9），鼓励路径多样性
> - **多轮反思（Reflexion）** → 高温度采样多条轨迹，选验证结果最优的一条
>
> **你的采样策略，就是 Agent 的"性格开关"。**

---



**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）离散和连续随机变量的本质区别？掷骰子、身高、点击/不点击各属哪类？
2. （概念）正态分布的 68-95-99.7 规则是什么？为什么异常检测用它？
3. （概念）MLE 的核心直觉，一句话回答。
4. （代码）模拟两个骰子之和的分布（10000 次），理论值 vs 模拟值柱状图对比。
5. （代码）对同一个 prompt，分别用 T=0.1 和 T=0.9 各生成 5 条模拟回复（用随机 logits + softmax 采样），对比两种温度下回复的多样性差异——统计每条回复中不同 token 的数量和重复率。
6. （代码）对 logits=[2.5,1.8,1.2,0.8,0.3,0.1,0.05,0.02] 用 greedy/temperature=0.7/temperature=1.5/top-k=4/top-p=0.85 五种策略各采样 2000 次，2×3 子图对比频率。
7. （代码）生成 μ=3, σ=1.5 的正态数据 100 个，MLE 估计 μ 和 σ。重复 100 次实验，画 μ̂ 分布直方图（应以 3 为中心）。
---


> 🔗 **章末钩子**：你用概率分布描述数据了。但事件不是孤立的——已知一件事发生，另一件事的概率怎么变？这需要"条件"。
> 预览：下一章——**条件概率与贝叶斯定理**，用已知信息更新判断。


> 💡 **提示**：完成后运行 Kernel -> Restart & Run All 验证所有代码块。
